# Workshop 4 - Pruebas cortas controladas de estabilidad

Controlador de `src/stability_experiments.py`. Las arquitecturas, perdidas, limites, metricas y comparaciones viven en scripts Python. El notebook se entrega sin ejecuciones ni resultados.

El AE z32 historico termino 30 epocas. Conservar best.pt de epoca 8, last.pt de epoca 30 y todos sus logs. Este protocolo crea modelos nuevos e independientes; el checkpoint opcional se usa solo para evaluacion y nunca para restaurar Adam.

1. Seleccionar un runtime GPU T4 ocioso; no ejecutar junto a otro entrenamiento.
2. Montar Drive y extraer el ZIP nuevo en una carpeta nueva de /content.
3. Revisar rutas, variantes y el plan sin --execute.
4. Habilitar EXECUTE solo despues de revisar el plan.

Una prueba corta sin errores no demuestra estabilidad durante 30 epocas. El control tanh es un cambio de arquitectura opcional, no una reparacion causal confirmada.


In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")
from pathlib import Path
from datetime import datetime, timezone
import json, subprocess, sys, zipfile


In [ ]:
# @title Rutas de codigo, datos y resultados
ZIP_SOURCE = "Drive" # @param ["Drive", "Upload"]
PROJECT_ZIP = "/content/drive/MyDrive/Workshop_4/workshop4_colab_stability_plan.zip" # @param {type:"string"}
EXTRACT_DIR = "/content/workshop4_stability_project" # @param {type:"string"}
DRIVE_PROCESSED_DIR = "/content/drive/MyDrive/Workshop_4/data/processed" # @param {type:"string"}
COPY_TRAIN_VAL_TO_LOCAL = True # @param {type:"boolean"}
LOCAL_PROCESSED_DIR = "/content/workshop4_stability_data/processed" # @param {type:"string"}
OUTPUT_PARENT = "/content/drive/MyDrive/Workshop_4/short_stability" # @param {type:"string"}
RUN_LABEL = "controlled_ae" # @param {type:"string"}
# Opcional: ruta a una copia ESTATICA de best.pt; lectura y evaluacion unicamente.
REFERENCE_CHECKPOINT = "" # @param {type:"string"}


In [ ]:
if ZIP_SOURCE == "Upload":
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Subir exactamente un ZIP del proyecto.")
    PROJECT_ZIP = next(iter(uploaded))
zip_path = Path(PROJECT_ZIP).expanduser().resolve()
if not zip_path.is_file():
    raise FileNotFoundError(zip_path)
with zipfile.ZipFile(zip_path) as archive:
    helpers = [name for name in archive.namelist()
               if name.endswith("Workshop_4/src/colab_utils.py")]
    if len(helpers) != 1:
        raise ValueError("El ZIP debe contener un unico Workshop_4/src/colab_utils.py")
    zip_import_root = helpers[0].removesuffix("src/colab_utils.py")
sys.path.insert(0, str(zip_path) + "/" + zip_import_root)
from src.colab_utils import extract_project_zip, stage_processed_parquet, runtime_info
WORKSHOP_DIR = extract_project_zip(zip_path, EXTRACT_DIR)
if not (WORKSHOP_DIR / "src/stability_experiments.py").is_file():
    raise FileNotFoundError("El ZIP no contiene el modulo de estabilidad actualizado")
PROCESSED_DIR = stage_processed_parquet(
    DRIVE_PROCESSED_DIR, LOCAL_PROCESSED_DIR if COPY_TRAIN_VAL_TO_LOCAL else None
)
print("Codigo:", WORKSHOP_DIR)
print("Train/val:", PROCESSED_DIR)
print("Configuracion:", WORKSHOP_DIR / "config/stability_experiments.yaml")


In [ ]:
# Solo comprueba el entorno; no instala ni sustituye el PyTorch de Colab.
info = runtime_info()
print(info)
if not info["cuda_available"]:
    print("GPU no disponible. Selecciona un runtime GPU o elige CPU explicitamente.")


## Diseno de los controles

- `reference_b128`: referencia sin cambios (batch 128, lr 0.001, z sin cota).
- `batch256` y `batch512`: cambian solamente batch.
- `lr0003`: cambia solamente learning rate a 0.0003, con batch 128.
- `latent_tanh`: cambia solamente la transformacion del latente; requiere inclusion explicita como arquitectura experimental.

`updates` compara el mismo numero de actualizaciones de Adam y registra cuantos ejemplos procesa cada batch. `examples` compara el mismo numero de ejemplos y registra cuantas actualizaciones requiere cada variante. `both` crea trials separados para ambos presupuestos; no une dos configuraciones dentro de un entrenamiento. Todos usan las mismas particiones y semilla, un modelo nuevo y un Adam nuevo.

Los limites del YAML incluyen tiempo por trial y total, actualizaciones y ejemplos totales. Los criterios de NaN/Inf, gradientes/latentes extremos y deterioro abortan de forma explicita. Ninguna variante puede convertir esta prueba en un entrenamiento de 30 epocas.


In [ ]:
# @title Seleccion explicita de las pruebas
TRIAL_NAMES = "reference_b128,batch256,batch512,lr0003" # @param {type:"string"}
BUDGET = "both" # @param ["updates", "examples", "both"]
DEVICE = "cuda" # @param ["cuda", "cpu", "auto"]
INCLUDE_ARCHITECTURE = False # @param {type:"boolean"}
MAX_SECONDS_PER_TRIAL = 120.0 # @param {type:"number"}
MAX_TOTAL_SECONDS = 600.0 # @param {type:"number"}
EXECUTE = False # @param {type:"boolean"}

selected = [name.strip() for name in TRIAL_NAMES.split(",") if name.strip()]
if not selected:
    raise ValueError("Seleccionar al menos una variante explicita")
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
if not RUN_LABEL or any(char in RUN_LABEL for char in "/\\"):
    raise ValueError("RUN_LABEL debe ser un nombre simple")
OUTPUT_DIR = Path(OUTPUT_PARENT).expanduser().resolve() / f"{RUN_LABEL}_{run_id}"
if OUTPUT_DIR.exists():
    raise FileExistsError(OUTPUT_DIR)
command = [
    sys.executable, "-m", "src.stability_experiments",
    "--config", str(WORKSHOP_DIR / "config/stability_experiments.yaml"),
    "--processed-dir", str(PROCESSED_DIR), "--device", DEVICE,
    "--budget", BUDGET, "--output-dir", str(OUTPUT_DIR),
    "--max-seconds-per-trial", str(MAX_SECONDS_PER_TRIAL),
    "--max-seconds-total", str(MAX_TOTAL_SECONDS),
]
for name in selected:
    command.extend(["--variant", name])
if INCLUDE_ARCHITECTURE:
    command.append("--include-architecture")
if REFERENCE_CHECKPOINT.strip():
    command.extend(["--reference-checkpoint", str(Path(REFERENCE_CHECKPOINT).expanduser().resolve())])
print("Salida NUEVA:", OUTPUT_DIR)


In [ ]:
# Vista del plan: no crea modelos entrenados ni escribe resultados.
print("Plan:", command)
subprocess.run(command, cwd=WORKSHOP_DIR, check=True)


In [ ]:
if EXECUTE:
    subprocess.run(command + ["--execute"], cwd=WORKSHOP_DIR, check=True, timeout=MAX_TOTAL_SECONDS + 30)
else:
    print("Prueba deshabilitada. Revisar el plan y activar EXECUTE para comenzar.")


## Interpretar los resultados

Las comparaciones deben leerse dentro del mismo presupuesto, con sus contadores reales de ejemplos y actualizaciones. Un trial abortado o limitado por tiempo no equivale a uno que completo su presupuesto. Revisar reconstruccion train/val, norma pre/post clipping, porcentaje recortado, magnitud de z, saturacion y sensibilidad del decoder, velocidad end-to-end y VRAM pico.

Una mejora de ejemplos/s medida durante cinco lotes no garantiza tiempo por epoca ni convergencia. El objetivo es elegir la siguiente configuracion para un experimento nuevo; no sobrescribir best.pt, last.pt ni interpretar estos trials como el entrenamiento final del taller.


In [ ]:
# Solo lee archivos que el modulo haya creado; no inventa resultados ausentes.
summary_path = OUTPUT_DIR / "summary.json"
if summary_path.is_file():
    print(summary_path.read_text(encoding="utf-8"))
else:
    print("No hay summary.json en esta salida (no se ha ejecutado o revisar el estado del trial).")
if OUTPUT_DIR.exists():
    print("CSV persistentes:")
    for path in sorted(OUTPUT_DIR.rglob("*.csv")):
        print(path.relative_to(OUTPUT_DIR))
else:
    print("Aun no existe la carpeta de resultados; el plan no escribe resultados.")


Los archivos nuevos de la prueba quedan en OUTPUT_DIR dentro de Drive. Conservar tambien la configuracion efectiva, el motivo de parada y el ZIP utilizado. Las instrucciones y limites se documentan en `STABILITY_PLAN.md`. Las fuentes historicas permanecen en sus carpetas originales; el checkpoint de referencia se evalua en lectura y su optimizador no se restaura.
